# Tarea 3 Inteligencia Artificial - Procesamiento del lenguaje natural

**Integrantes:**
*   Guillermo Gonzales
*   Juan de Dios Godoy
*   Cristobal Salgado
*   Ignacio Vidal


In [1]:
#!pip install spacy #Instalamos spaCy para poder lematizar las oraciones
#!python -m spacy download en_core_web_sm #descargamos el modelo pre entrenado en ingles
#!python -m spacy download es_core_web_sm #descargamos el modelo en español
#!python -m spacy download fr_core_web_sm #descargamos el modelo en frances
#!pip install langdetect #Para determinar los idiomas del dataframe

In [2]:
#Importamos librerias necesarias
import numpy as np
import pandas as pd
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from collections import Counter

import spacy
import en_core_web_sm
from string import punctuation
from spacy.lang.es.stop_words import STOP_WORDS as stop_words_es
from spacy.lang.en.stop_words import STOP_WORDS as stop_words_en
from spacy.lang.fr.stop_words import STOP_WORDS as stop_words_fr

from gensim.models import Word2Vec

from scipy.spatial.distance import cosine

### Importamos la base de datos y realizamos la limpieza necesaria del modelo

In [3]:
#Cargamos el archivo a utilizar
df = pd.read_excel('Corpus-Agro.xlsx', engine='openpyxl')
print("El tamaño de la base de datos es:", df.shape)
print("El tipo de dato de resumen es:", df['Resumen'].dtype)
df.head()

El tamaño de la base de datos es: (100908, 2)
El tipo de dato de resumen es: object


,URL de Documento,Resumen
0,https://faolex.fao.org/docs/pdf/vie78248.pdf,"This Decree provides for the functions, tasks,..."
1,https://faolex.fao.org/docs/pdf/vie95279.pdf,This Decision approves the Scheme on the devel...
2,https://faolex.fao.org/docs/pdf/ita121539.pdf,This Regional Act sets out the legislative fra...
3,https://faolex.fao.org/docs/pdf/cro126979.pdf,This Regulation amends various provisions of t...
4,https://faolex.fao.org/docs/pdf/chn137120C.pdf...,This Law is enacted for the purposes of guaran...


In [4]:
for i in range(5):
  print(df.iloc[i,1])
  print("****")

This Decree provides for the functions, tasks, powers and organizational structure of the Ministry of Agriculture and Rural Development. The Ministry of Agricultural and Rural Development is a governmental agency which shall perform function of state management in the domains of agriculture, plant protection, animal health, food quality, forestry, salt-making, fisheries, irrigation, rural development, etc. Tasks and powers of the Ministry are specified in the text. 
****
This Decision approves the Scheme on the development of agricultural and forest plant varieties, livestock breeds and aquatic strains. The main objective of the Scheme is to raise the capacity of the system of research, selection, creation, transfer, production and supply of cultivation plant varieties, livestock breeds, forest tree varieties and aquatic strains in order to rapidly raise the yield, quality, competitiveness and effectiveness of agricultural production, forestry and fisheries. The Decision further provid

La siguiente linea se utilizo para realizar testeos con un dataset mas pequeño

In [5]:
#Truncamos a 1000 datos
#df = df.sample(n=100, random_state=42).reset_index(drop=True)
#df.head()

Dado que tiene muchos idiomas generamos un detector de idioma

In [6]:
#Asegúrate de que los resultados sean consistentes en cada ejecución
DetectorFactory.seed = 0

# Función para detectar el idioma
def detectar_idioma(resumen):
    if isinstance(resumen, str) and resumen.strip():  # Comprobar si es un string no vacío
        try:
            # Detectar el idioma del resumen
            return detect(resumen)
        except Exception:
            return "Desconocido"  # Si hay un error, devuelve "Desconocido"
    else:
        return "Desconocido"  # Si el resumen no es válido, devuelve "Desconocido"

# Aplicar la función para crear una nueva columna de idioma
df['Idioma'] = df['Resumen'].apply(detectar_idioma)

# Mostrar el DataFrame con la nueva columna
print(df[['Resumen', 'Idioma']].head())

                                             Resumen Idioma
0  This Decree provides for the functions, tasks,...     en
1  This Decision approves the Scheme on the devel...     en
2  This Regional Act sets out the legislative fra...     en
3  This Regulation amends various provisions of t...     en
4  This Law is enacted for the purposes of guaran...     en


In [7]:
# Función para lematizar el texto según el idioma
def Lematizar(oracion, idioma):
    oracion = str(oracion)
    if idioma == 'en':
        doc = nlp_en(oracion)
    elif idioma == 'es':
        doc = nlp_es(oracion)
    elif idioma == 'fr':
        doc = nlp_fr(oracion)
    else:
        return oracion
    
    # Lematización, pasando a minúsculas y eliminando stopwords y puntuación
    lemas = [
        token.lemma_.lower().strip() for token in doc
        if token.text not in stop_words[idioma] and token.text not in punctuations
    ]
    return " ".join(lemas)


La aplicacion de la funcion a los resumenes es lo que realiza la lematizacion a cada resumen de todo el dataset y es uno de los 2 bloques que se demoran mas o menos 50 min

In [8]:
# Cargar los modelos de spaCy para los tres idiomas
nlp_en = spacy.load('en_core_web_sm')
nlp_es = spacy.load('es_core_news_sm')
nlp_fr = spacy.load('fr_core_news_sm')

# Definir stopwords y puntuación
stop_words = {
    'en': set(stop_words_en),
    'es': set(stop_words_es),
    'fr': set(stop_words_fr),
}
punctuations = set(punctuation)

# Aplicar la función a cada resumen
df['Resumen_lem'] = df.apply(lambda row: Lematizar(row['Resumen'], row['Idioma']), axis=1)

In [9]:
#imprimimos todos isiomas detectados y los que no se reconocieron como desconocidos
print("Idiomas:", df['Idioma'].unique())
df.head()

Idiomas: ['en' 'es' 'fr' 'Desconocido' 'ca' 'sl']


,URL de Documento,Resumen,Idioma,Resumen_lem
0,https://faolex.fao.org/docs/pdf/vie78248.pdf,"This Decree provides for the functions, tasks,...",en,this decree provide function task power organi...
1,https://faolex.fao.org/docs/pdf/vie95279.pdf,This Decision approves the Scheme on the devel...,en,this decision approve scheme development agric...
2,https://faolex.fao.org/docs/pdf/ita121539.pdf,This Regional Act sets out the legislative fra...,en,this regional act set legislative framework go...
3,https://faolex.fao.org/docs/pdf/cro126979.pdf,This Regulation amends various provisions of t...,en,this regulation amend provision regulation imp...
4,https://faolex.fao.org/docs/pdf/chn137120C.pdf...,This Law is enacted for the purposes of guaran...,en,this law enact purpose guarantee agricultural ...


La siguiente fila (18074) generaba conflicto dado que se encontraba vacia, era la unica en todo el dataset que estaba vacia, por lo que debemos trabaajarla mas adelante

In [10]:
# Imprimir las filas desde el índice 18070 hasta el 18079
print(df.iloc[18074])


URL de Documento    https://faolex.fao.org/docs/pdf/pan127522.pdf;...
Resumen                                                          . . 
Idioma                                                    Desconocido
Resumen_lem                                                      . . 
Name: 18074, dtype: object


### Ahora que filtramos todos los datos, les asignamos un idioma, procedemos a tokenizarlo, que es ponerlo de una forma que el modelo pueda vectorizarlo para poder posteriormente predecir las aproximaciones de las consultas realizadas

In [11]:
# Tokenizar cada resumen para Word2Vec
# Esta línea crea una lista de listas donde cada sublista contiene las palabras de un resumen lematizado.
corpus_tokenizado = [str(resumen).split() for resumen in df['Resumen_lem']]

# Entrenar el modelo
# Esta línea entrena un modelo de Word2Vec utilizando el corpus tokenizado.
# - sentences: es el corpus que contiene las oraciones tokenizadas.
# - vector_size: determina la dimensionalidad de los vectores de palabras generados.
# - window: define el tamaño del contexto que se considera para las palabras (número de palabras a la izquierda y derecha de la palabra objetivo).
# - min_count: establece el número mínimo de veces que una palabra debe aparecer en el corpus para ser considerada.
# - workers: especifica el número de hilos de CPU a utilizar durante el entrenamiento para mayor velocidad.
modelo_w2v = Word2Vec(sentences=corpus_tokenizado, vector_size=100, window=5, min_count=2, workers=4)

# Guardar el modelo entrenado para usarlo después
# Esta línea guarda el modelo entrenado en un archivo con el nombre especificado.
# El modelo se podrá cargar más tarde sin necesidad de volver a entrenarlo, ahorrando tiempo y recursos.
modelo_w2v.save("modelo_word2vec.model")


Vectoriza la consulta (convierte la pregunta en vector, un arreglo de 100 de largo con puros numeros)

In [12]:
# Función para vectorizar la consulta
def vectorizar_consulta(consulta):
    # Detectar el idioma
    try:
        idioma = detect(consulta)
    except Exception as e:
        print("Error al detectar el idioma:", e)
        idioma = 'en'  # Asignar inglés como idioma predeterminado en caso de error

    # Si el idioma detectado no está en la lista, asignar "en"
    if idioma not in ['en', 'es', 'fr']:
        print("Idioma no soportado, usando 'en' por defecto.")
        idioma = 'en'

    # Lematizar la consulta según el idioma detectado
    lematizar_consulta = Lematizar(consulta, idioma).split()
    consulta_tokenizada = [modelo_w2v.wv[consulta_token] for consulta_token in lematizar_consulta if consulta_token in modelo_w2v.wv]

    # Calcular y devolver el vector de la consulta
    return np.mean(consulta_tokenizada, axis=0)

Funcion que vectoriza los resumenes del dataset para posteriormente compararlo con la pregunta aqui se resuelven los conflictos, en caso de tener un dioma que no sean los 3 predefinidos (es en fr) se asigna la vectorizacion automaticamente en ingles (en) en el caso de estar el bloque del dataset vacio, se asigna con el vector (0)

In [13]:
# Función para vectorizar el resumen
def vectorizar_resumen(resumen):
    # Detectar el idioma
    try:
        idioma = detect(resumen)
    except Exception as e:
        print("Error al detectar el idioma:", e)
        idioma = 'en'  # Asignar inglés como idioma predeterminado en caso de error

    # Si el idioma detectado no está en la lista, asignar "en"
    if idioma not in ['en', 'es', 'fr']:
        print("Idioma no soportado, usando 'en' por defecto.")
        idioma = 'en'

    # Lematizar el resumen según el idioma detectado
    lematizar_resumen = Lematizar(resumen, idioma).split()
    
    # Obtener vectores de las palabras lematizadas
    resumen_tokenizado = []
    for resumen_token in lematizar_resumen:
        if resumen_token in modelo_w2v.wv:
            resumen_tokenizado.append(modelo_w2v.wv[resumen_token])
    
    # Verificar si se encontraron vectores
    if len(resumen_tokenizado) == 0:
        print("No se encontraron vectores para el resumen. Resumen omitido.")
        return None  # Retornar None en lugar de un vector cero si no hay vectores
    
    # Calcular y devolver el vector del resumen
    return np.mean(resumen_tokenizado, axis=0)
    

Este e el segundo bloque que se demora en promedio 50 min, ya que es el que genera la nueva columna con los resumenes ya lematizados y vectorizados

In [14]:
df['Vector_resumen'] = df['Resumen_lem'].apply(vectorizar_resumen)

Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' por defecto.
Idioma no soportado, usando 'en' p

Aqui guardamos el dataset con ya todo ajustado para evitar tener que hacer todo otra vez y poder ejecutar el modelo sin necesidad que tarde 2 horas

In [15]:
# Guardar el DataFrame en un nuevo archivo de Excel
df.to_excel("Corpus_Procesado.xlsx", index=False)

Este salio por las pruebas para determinar cual era el que generaba conflicto, pero se resolvio conviertiendo el resumen vacio en el vector de 0

In [16]:
# Imprime la fila con el índice 18074
print(df.loc[18074])
#300


URL de Documento    https://faolex.fao.org/docs/pdf/pan127522.pdf;...
Resumen                                                          . . 
Idioma                                                    Desconocido
Resumen_lem                                                      . . 
Vector_resumen                                                   None
Name: 18074, dtype: object


funcion del buscador que genera la similitud con la consulta 

In [17]:
def buscador(consulta, n=5):
    vector_consulta = vectorizar_consulta(consulta)
    
    # Imprimir el vector de consulta para depuración (pruebas de testeo)
    #print(f"Vector de consulta: {vector_consulta}")
    #print(f"Forma del vector de consulta: {vector_consulta.shape if hasattr(vector_consulta, 'shape') else 'No tiene forma'}")
    
    # Verificar si el vector de consulta es válido
    if vector_consulta is None or len(vector_consulta) == 0:
        print("El vector de consulta es vacío o nulo.")
        return "La consulta no generó un vector válido."

    # Calcular la similitud si es None se lo salta
    df['Similitud'] = df['Vector_resumen'].apply(lambda x: 1 - cosine(vector_consulta, x) if x is not None else -1)
    result = df.nlargest(n, 'Similitud')

    return result[['URL de Documento', 'Resumen', 'Similitud']]


### consulta y similitud

ejemplo de uso con la 4 linea de la base de datos seleccionando su resumen

In [20]:
consulta = df['Resumen'][4]
buscador(consulta, 7)

,URL de Documento,Resumen,Similitud
4,https://faolex.fao.org/docs/pdf/chn137120C.pdf...,This Law is enacted for the purposes of guaran...,0.998450
90677,https://faolex.fao.org/docs/pdf/chn137163.pdf,The purpose of these Measures is to ensure the...,0.979780
4440,https://faolex.fao.org/docs/pdf/chn64841.pdf,The purpose of this Law is to guarantee the qu...,0.976802
92314,https://faolex.fao.org/docs/pdf/chn188625.pdf,"These Measures, consisting of 13 Articles, are...",0.976055
98406,https://faolex.fao.org/docs/pdf/chn214985.pdf,This Law is enacted with a view to reinforcing...,0.974745
82740,https://faolex.fao.org/docs/pdf/chn139822.pdf,The purpose of these Regulations is to regulat...,0.973832
4902,https://faolex.fao.org/docs/pdf/bra170883.pdf,"This Norm, consisting of 20 articles and one A...",0.973359


ejemplo de uso con palabra cualquiera, se puede indicar maualmente cambiando la seccion comentada, pero para este caso demostraremos con la palabra "manzana", la cual no tiene relacion con los resumenes y dado la poca coincidencia nos marcara gran similitud con el dato que estava vacio, por lo que si queremos efectuar una verdadera busqueda consideremos desde el segundo

In [21]:
#consulta = input("Ingrese su consulta: ")
consulta = "manzana"
buscador(consulta, 7)

Idioma no soportado, usando 'en' por defecto.


,URL de Documento,Resumen,Similitud
54430,https://faolex.fao.org/docs/pdf/arg3776.pdf,"Se refiere a las siguientes especies: alfalfa,...",0.907822
55594,https://faolex.fao.org/docs/pdf/mex17958.pdf,Esta Norma tiene por objeto establecer los lin...,0.899913
61447,https://faolex.fao.org/docs/pdf/ecu102116.pdf,La presente Resolución establece los requisito...,0.896643
25262,https://faolex.fao.org/docs/pdf/per85875.pdf,La presente Resolución suspende por un período...,0.892650
87407,https://faolex.fao.org/docs/pdf/ecu109654.pdf,La presente Resolución establece los requisito...,0.891199
26805,https://faolex.fao.org/docs/pdf/per105347.pdf,La presente Resolución suspende temporalmente ...,0.890233
24287,https://faolex.fao.org/docs/pdf/per73670.pdf,La presente Resolución suspende temporalmente ...,0.890100
